# 02 — Feature Engineering: account/window + graph features

Thin walkthrough calling into `src/features.py` and `src/graph_features.py` (PLAN.md Phase 3) — logic lives in `src/`, this notebook just runs it, inspects the output, and saves the modelling table to `data/processed/`. Both modules are leakage-safe by construction: account/window features use rolling stats bounded by each transaction's own timestamp, and graph features are built from a per-calendar-day snapshot using only strictly-prior days' edges. See `tests/test_features.py` and `tests/test_graph_features.py` for the explicit leakage assertions.

In [1]:
import sys
import time
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd

from src.data_loader import load_config, load_transactions
from src.features import build_modelling_table

config = load_config("../config.yaml")
raw_dir = Path("..") / config["paths"]["raw_dir"]

In [2]:
txns = load_transactions(raw_dir / "HI-Small_Trans.csv")
print(f"transactions: {len(txns):,}")

transactions: 5,078,345


## Build the modelling table

Joins `assemble_feature_table` (account/window velocity, volume, structuring score, pass-through ratio — `src/features.py`) with `build_daily_graph_features` (degree, fan-in/fan-out score, cycle membership — `src/graph_features.py`). On the full HI-Small dataset (~5.08M rows) this takes on the order of 15-20 minutes single-threaded — a one-time offline pipeline step per CLAUDE.md's deployment model, never run inside the live Streamlit app.

In [3]:
t0 = time.time()
features = build_modelling_table(txns, config)
elapsed = time.time() - t0
print(f"Modelling table: {features.shape[0]:,} rows, {features.shape[1]} columns in {elapsed:.1f}s")

Modelling table: 5,078,345 rows, 52 columns in 1531.6s


In [4]:
features.columns.tolist()

['timestamp',
 'from_account_key',
 'to_account_key',
 'amount_paid_usd',
 'payment_format',
 'is_laundering',
 'sender_out_1d_count',
 'sender_out_1d_amount_usd',
 'sender_out_7d_count',
 'sender_out_7d_amount_usd',
 'sender_out_30d_count',
 'sender_out_30d_amount_usd',
 'sender_in_1d_count',
 'sender_in_1d_amount_usd',
 'sender_in_7d_count',
 'sender_in_7d_amount_usd',
 'sender_in_30d_count',
 'sender_in_30d_amount_usd',
 'receiver_out_1d_count',
 'receiver_out_1d_amount_usd',
 'receiver_out_7d_count',
 'receiver_out_7d_amount_usd',
 'receiver_out_30d_count',
 'receiver_out_30d_amount_usd',
 'receiver_in_1d_count',
 'receiver_in_1d_amount_usd',
 'receiver_in_7d_count',
 'receiver_in_7d_amount_usd',
 'receiver_in_30d_count',
 'receiver_in_30d_amount_usd',
 'sender_out_1d_distinct_counterparties',
 'sender_out_7d_distinct_counterparties',
 'sender_out_30d_distinct_counterparties',
 'receiver_in_1d_distinct_counterparties',
 'receiver_in_7d_distinct_counterparties',
 'receiver_in_30d_di

In [5]:
features.describe().T

,count,mean,min,25%,50%,75%,max,std
timestamp,5078345,2022-09-05 07:16:08.194277120,2022-09-01 00:00:00,2022-09-02 04:32:00,2022-09-05 12:16:00,2022-09-08 03:13:00,2022-09-18 16:18:00,NaN
amount_paid_usd,5078345.0,343239.987684,0.000071,152.13,860.697625,5138.67,28493009546.476501,24411566.053543
is_laundering,5078345.0,0.001019,0.0,0.0,0.0,0.0,1.0,0.031912
sender_out_1d_count,5078345.0,851.126477,1.0,2.0,5.0,9.0,26368.0,3588.355379
sender_out_1d_amount_usd,5078345.0,265846031.96114,0.000071,1840.72,14491.15,217291.73424,28493009546.476501,1383242257.867341
sender_out_7d_count,5078345.0,3711.325945,1.0,5.0,16.0,38.0,120029.0,16809.120901
sender_out_7d_amount_usd,5078345.0,1262807846.824965,0.000071,13119.4639,132379.4,1313940.7,43912941622.725044,5563608358.151563
sender_out_30d_count,5078345.0,4129.297828,1.0,5.0,17.0,42.0,168672.0,19300.255293
sender_out_30d_amount_usd,5078345.0,1502646306.670485,0.000071,17917.61,181450.2131,1634175.52,52762291209.75,6712884313.315598
sender_in_1d_count,5078345.0,6.980155,0.0,0.0,2.0,3.0,539.0,38.775025


## Sanity checks

No NaNs (every rolling stat has a well-defined cold-start default of 0/False), and laundering rows should show visibly different structuring/graph signal than legitimate rows on average — a first sniff test that the features carry signal before any model sees them.

In [6]:
assert not features.isna().any().any(), "unexpected NaNs in the modelling table"
print("No NaNs: OK")

No NaNs: OK


In [7]:
signal_cols = [
    "structuring_score_7d",
    "sender_out_7d_distinct_counterparties",
    "receiver_in_7d_distinct_counterparties",
    "sender_graph_in_cycle",
    "sender_pass_through_ratio_1d",
]
features.groupby("is_laundering")[signal_cols].mean()

,structuring_score_7d,sender_out_7d_distinct_counterparties,receiver_in_7d_distinct_counterparties,sender_graph_in_cycle,sender_pass_through_ratio_1d
is_laundering,,,,,
0,37.403052,597.208122,2.845173,0.439914,1274.281519
1,51.872899,838.724937,4.339772,0.556693,30.720583


## Save the modelling table

Written to `data/processed/` (gitignored — regenerable from this notebook / `src/features.py`'s `__main__` block, per CLAUDE.md).

In [8]:
out_path = Path("..") / config["paths"]["modelling_table"]
out_path.parent.mkdir(parents=True, exist_ok=True)
features.to_parquet(out_path)
print(f"Saved {len(features):,} rows to {out_path}")

Saved 5,078,345 rows to ..\data\processed\features.parquet
